# Test — Reference Book Ingestion (standalone)

Exercises **only the reference KB pipeline**: PDF → chunk → **local GPU
embeddings** → persistent **ChromaDB** → similarity search. No LLM / OpenAI key
needed (ingestion + search are fully local).

Run this to confirm the two threat-modeling books index correctly and that the
GPU is used.

## Prerequisites

Run from the project root (`ai-threat-modeling-assistant/`):

```bash
pip install -r requirements.txt
# RTX 5090 (Blackwell) needs a CUDA 12.8+ torch build:
pip install torch --index-url https://download.pytorch.org/whl/cu128
```

The books must be in a `Reference/Books` folder (auto-discovered upward) or set
`REFERENCE_BOOKS_DIR`. First run downloads the embedding model (~130 MB).

In [1]:
import logging
from dotenv import load_dotenv
import threat_model.references as R

load_dotenv()
logging.basicConfig(level="INFO", format="%(asctime)s | %(levelname)s | %(message)s")

books = R.resolve_books_dir()
print("Books dir       :", books)
print("Embedding model :", R.EMBEDDING_MODEL)
print("Device (auto)   :", R._device())
print("Chroma dir      :", R.CHROMA_DIR)
print("Collection      :", R.COLLECTION)
print("Chunk size/ovlp :", R.CHUNK_SIZE, "/", R.CHUNK_OVERLAP)
assert books, "No books found — set REFERENCE_BOOKS_DIR or add PDFs to Reference/Books"


Books dir       : /workspaces/VS/Reference/Books
Embedding model : BAAI/bge-small-en-v1.5
Device (auto)   : cuda
Chroma dir      : /workspaces/VS/MasteringAgenticAI/Week 1/ai-threat-modeling-assistant/data/reference_chroma
Collection      : threat_modeling_refs
Chunk size/ovlp : 900 / 150


## 1. Confirm GPU is available

The embedder auto-selects CUDA when present (your RTX 5090). If this prints
`False`, ingestion still works on CPU (slower).

In [2]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


CUDA available: True
GPU: NVIDIA GeForce RTX 5090


## 2. Parse preview (fast, no embeddings)

Load + chunk each PDF without embedding, to sanity-check parsing and chunk counts.
Note: some "PDFs" are saved HTTP multipart uploads; the loader strips that wrapper
automatically.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=R.CHUNK_SIZE, chunk_overlap=R.CHUNK_OVERLAP)
for pdf in sorted(books.glob("*.pdf")):
    chunks = R._load_pdf_chunks(pdf, splitter)
    pages = max((c["metadata"]["page"] for c in chunks), default=0)
    print(f"{pdf.name}: {len(chunks)} chunks across ~{pages} text-pages")
    if chunks:
        print("   sample:", chunks[0]["id"], "|", chunks[0]["metadata"])


/home/vscode/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


9781801076531.pdf: 1345 chunks across ~392 text-pages
   sample: 9781801076531-p0002-c00 | {'source': '9781801076531', 'source_file': '9781801076531.pdf', 'page': 2, 'chunk_index': 0}
9781836648635.pdf: 1846 chunks across ~564 text-pages
   sample: 9781836648635-p0002-c00 | {'source': '9781836648635', 'source_file': '9781836648635.pdf', 'page': 2, 'chunk_index': 0}


2026-06-09 22:36:30,322 | INFO | threat_modeling_designing_for_security.pdf: stripped non-PDF wrapper (PDF starts at byte 198)


AutomotiveThreatModeling.pdf: 869 chunks across ~274 text-pages
   sample: AutomotiveThreatModeling-p0001-c00 | {'source': 'AutomotiveThreatModeling', 'source_file': 'AutomotiveThreatModeling.pdf', 'page': 1, 'chunk_index': 0}
threat_modeling_designing_for_security.pdf: 1932 chunks across ~627 text-pages
   sample: threat_modeling_designing_for_security-p0002-c00 | {'source': 'threat_modeling_designing_for_security', 'source_file': 'threat_modeling_designing_for_security.pdf', 'page': 2, 'chunk_index': 0}


## 3. Ingest into ChromaDB (uses the GPU)

`ingest()` is **incremental** (per-file content hash → `data/reference_index_state.json`):
new/changed PDFs are embedded and stored, unchanged ones are skipped. Use
`force=True` to re-index everything.

In [4]:
kb = R.ReferenceKB()
report = kb.ingest()          # PDF -> chunk -> GPU embed -> ChromaDB
print("Ingest report:", report)
print("Total chunks in collection:", kb.count())


2026-06-09 22:36:55,181 | INFO | Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
2026-06-09 22:36:55,266 | INFO | Reading 9781801076531.pdf…
2026-06-09 22:37:00,632 | INFO | Embedding 1345 chunks from 9781801076531.pdf…
2026-06-09 22:37:00,633 | INFO | Loading reference embedding model 'BAAI/bge-small-en-v1.5' on device 'cuda'…
2026-06-09 22:37:00,783 | INFO | HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-06-09 22:37:00,784 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-06-09 22:37:00,815 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
2026-06-09 22:37:00,914 | INFO | HTTP Request: HEAD https://huggingface.co/BAA

Ingest report: {'files_indexed': 2, 'files_skipped': 2, 'chunks': 3191}
Total chunks in collection: 5992


## 4. Sample similarity searches

Confirm retrieval returns relevant passages with **source + page** metadata.

In [5]:
queries = [
    "spoofing an API client to bypass authentication",
    "STRIDE threats for a data store",
    "mitigations for tampering with messages in transit",
]
for q in queries:
    print("=" * 90, "\nQUERY:", q)
    for h in kb.search(q, k=3):
        m = h["metadata"]
        print(f"  score={h['score']:.3f}  {m.get('source')} p.{m.get('page')}")
        print("   ", h["text"][:220].replace("\n", " "), "…")


QUERY: spoofing an API client to bypass authentication


Batches: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 121.59it/s]


  score=0.766  threat_modeling_designing_for_security p.538
    which your authentication code relies, and the attacker can use whatever attack  tools they’d like. Resisting online attacks involves backoff and possibly lockout,  while resisting offl ine attacks involves salts and ite …
  score=0.757  threat_modeling_designing_for_security p.110
    This extended example discusses how STRIDE threats could manifest against  the Acme/SQL database described in Chapter 1, “Dive In and Threat Model!”  and 2, “Strategies for Threat Modeling,” and shown in Figure 2-1. You’ …
  score=0.748  threat_modeling_designing_for_security p.100
    64 Part II ■ Finding Threats c03.indd 07:51:54:AM  01/15/2014 Page 64 by putting a program with that name on disk. You can spoof an endpoint on  the same machine by squatting or splicing. You can spoof users by capturing …
QUERY: STRIDE threats for a data store


Batches: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 19.14it/s]


  score=0.833  threat_modeling_designing_for_security p.608
    from data stores:  STRIDE threat tree,  459–462 bypassing protection  subtree, 460–461 diagram, 459 metadata and side  channels subtree,  461 storage attacks subtree,  462 STRIDE-per-Element  diagram, 431 surprise subtre …
  score=0.820  threat_modeling_designing_for_security p.12
    Elevate Privileges through Authorization Failures 74 Extended Example: STRIDE Threats against Acme-DB 74 STRIDE Variants 78 STRIDE-per-Element 78 STRIDE-per-Interaction 80 DESIST 85 Exit Criteria 85 Summary 85 Chapter 4  …
  score=0.820  threat_modeling_designing_for_security p.12
    x Contents ftoc.indd 11:56:7:AM  01/17 /2014 Page x Color in Diagrams 53 Entry Points 53 Validating Diagrams 54 Summary 56 Part II Finding Threats  59 Chapter 3 STRIDE 61 Understanding STRIDE and Why It’s Useful 62 Spoof …
QUERY: mitigations for tampering with messages in transit


Batches: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 147.80it/s]

  score=0.761  threat_modeling_designing_for_security p.470
    434 Appendix B ■ Threat Trees bapp02.indd 06:45:12:PM  01/20/2014 Page 434 Also consider whether tampering threats against the authorization process  should be in scope for your system. Table B-1a: Spooﬁ ng by Obtaining  …
  score=0.735  9781836648635 p.294
    vehicle by passively reading CAN bus traffic, without injecting messages, without compromising an ECU, and without triggering any observable anomaly in vehicle behavior [35]. 257 Attacking the In-Vehicle Communications …
  score=0.735  threat_modeling_designing_for_security p.103
    Chapter 3 ■ STRIDE 67 c03.indd 07:51:54:AM  01/15/2014 Page 67 Tampering Threats Tampering is modifying something, typically on disk, on a network, or in  memory. This can include changing data in a spreadsheet (using ei …


## 5. Incremental re-ingest

Running again should **skip** the already-indexed files (no re-embedding).

In [6]:
print("Re-running ingest (incremental):")
print(kb.ingest())
print("Total chunks:", kb.count())


2026-06-09 22:37:33,937 | INFO | Reference ingest done: {'files_indexed': 0, 'files_skipped': 4, 'chunks': 0} | total=5992


Re-running ingest (incremental):
{'files_indexed': 0, 'files_skipped': 4, 'chunks': 0}
Total chunks: 5992


## Notes

- This is the exact pipeline `service.generate_threat_model_report()` uses to
  ground generation: when the app is in **OpenAI mode**, it calls
  `references.retrieve_reference_context(description)` and injects the top passages
  into the prompt.
- Reset the index with `R.ReferenceKB().reset()` or `kb.ingest(force=True)`.
- CLI equivalent: `python ingest_references.py` (and `--stats`, `--search "..."`, `--force`).